# Optimización de carteras con InvestingLab

Esta demo parte de una cartera equiponderada y busca otros pesos con varios objetivos:
mínima varianza, máximo Sharpe, paridad de riesgo, HRP y vistas de Black-Litterman. Acaba
con la frontera eficiente y el informe de la cartera optimizada.

Los datos salen de las fuentes configuradas (por ejemplo, tus ficheros en
`INVESTINGLAB_LOCAL_DATA_DIR`). Todo se estima **en muestra**: los pesos optimizados parecen
mejores en el histórico usado de lo que serán después. Más detalle en
`docs/guide/optimization.md`.

In [ ]:
import pandas as pd

from investinglab.config import LogLevel, configure_logging
from investinglab.domain import View
from investinglab.toolkit import Toolkit

configure_logging(LogLevel.WARNING)
tk = Toolkit.from_settings()

SIMBOLOS = ["SAN.MC", "ITX.MC", "IBE.MC", "BBVA.MC"]
cartera = tk.portfolio("Núcleo", dict.fromkeys(SIMBOLOS, 0.25), start="2019-01-01")
cartera

## Mínima varianza y máximo Sharpe

`summary()` compara los pesos de partida con los optimizados, junto con la rentabilidad, la
volatilidad y la parte del riesgo de cada activo. El límite `max_weight` evita que un solo
activo se lleve casi todo el peso.

In [ ]:
minima = cartera.optimize()
tangente = cartera.optimize(objective="max_sharpe", max_weight=0.5, risk_free_rate=0.03)
tangente.summary()

In [ ]:
pd.Series(
    {
        "Volatilidad mínima": minima.volatility,
        "Sharpe de la tangente": tangente.sharpe_ratio,
        "Activos efectivos (tangente)": tangente.effective_number_of_assets,
    }
)

## Sin rentabilidades esperadas: paridad de riesgo y HRP

Las medias históricas son muy ruidosas. Estos objetivos solo usan la covarianza, así que son
más estables.

In [ ]:
objetivos = {
    "Mínima varianza": "min_variance",
    "Paridad de riesgo": "risk_parity",
    "HRP": "hrp",
    "Inversa de la volatilidad": "inverse_volatility",
}
comparativa = pd.DataFrame(
    {nombre: cartera.optimize(objective=objetivo).weights for nombre, objetivo in objetivos.items()}
)
comparativa

## Vistas (Black-Litterman)

Parte de las rentabilidades de equilibrio (las que harían óptima la cartera actual) y añade
dos opiniones: una absoluta y otra relativa. Cuanto menor es la incertidumbre, más pesa la
vista.

In [ ]:
con_vistas = cartera.optimize(
    objective="max_sharpe",
    expected_returns="equilibrium",
    risk_free_rate=0.03,
    views=[
        View.absolute("IBE.MC", 0.09),
        View.relative("ITX.MC", "SAN.MC", 0.03, uncertainty=0.0004),
    ],
)
con_vistas.summary()

## Frontera eficiente

Cada punto es la cartera de menor volatilidad para una rentabilidad esperada. El gráfico
marca los activos, la cartera de mínima varianza, la tangente y la línea del mercado de
capitales.

In [ ]:
frontera = cartera.efficient_frontier(points=40, risk_free_rate=0.03, max_weight=0.6)
frontera.plot()

## La cartera optimizada es una cartera más

`tangente.portfolio` conserva los activos, la divisa y el rebalanceo. Se analiza, se simula y
se documenta como cualquier otra cartera.

In [ ]:
informe = tk.report(tangente.portfolio)
informe.save("salidas/tangente.md")
informe

In [ ]:
tk.close()